In [ ]:
import json
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import wandb
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# 0) 재현성을 위한 시드 고정
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# ─────────────────────────────────────────────────────────────────────────────

# ==================== 1. Load & Preprocess ====================
with open('./util/result_clean.json','r') as f:
    raw = json.load(f)

rows = []
for uid, user in enumerate(raw):
    for ts, tok in enumerate(user['token_sequence']):
        rows.append((uid, tok, ts))
_df = pd.DataFrame(rows, columns=['user_id','item_id','timestamp'])
user_seqs = _df.groupby('user_id')['item_id'].apply(list).tolist()

# 이제 “길이 >= 3인 시퀀스만 사용” (leave-two-out을 적용하려면 최소 3개 이상 필요)
valid_seqs = [seq for seq in user_seqs if len(seq) >= 3]

# 토큰 ↔ ID 매핑
unique_items = sorted(_df['item_id'].unique())
token2id = {t: i+1 for i, t in enumerate(unique_items)}  # 0 = PAD
id2token = {i: t for t, i in token2id.items()}

# ==================== 2. Dataset 정의 (변경 없음) ====================
class SASRecDataset(Dataset):
    """
    Hold-out the last item for recommendation evaluation.
    (validation/test / target 단일 아이템 예측용)
    """
    def __init__(self, sequences, t2i, max_len=20):
        self.seqs    = sequences
        self.t2i     = t2i
        self.max_len = max_len

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        ids = [self.t2i[t] for t in self.seqs[idx] if t in self.t2i]
        ids = ids[-self.max_len:]
        prefix, target = ids[:-1], ids[-1]
        L = len(prefix)
        pad_len = self.max_len - L
        # input_ids: [PAD ... PAD] + [item1, item2, ..., item_L]
        input_ids = [0]*pad_len + prefix
        return (
            torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(L,         dtype=torch.long),
            torch.tensor(target,    dtype=torch.long)
        )

# ==================== 3. SASRec 모델 정의 (변경 없음) ====================
class SASRec(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.max_len     = cfg['max_seq_length']
        self.hidden_size = cfg['hidden_size']
        self.n_items     = cfg['n_items']

        # item embedding (padding_idx=0)
        self.item_emb = nn.Embedding(self.n_items+1, self.hidden_size, padding_idx=0)
        self.pos_emb  = nn.Embedding(self.max_len, self.hidden_size)

        enc = nn.TransformerEncoderLayer(
            d_model=self.hidden_size,
            nhead=cfg['n_heads'],
            dim_feedforward=cfg['inner_size'],
            dropout=cfg['hidden_dropout_prob'],
            activation=cfg['hidden_act'],
            layer_norm_eps=cfg['layer_norm_eps'],
            batch_first=True
        )
        self.encoder     = nn.TransformerEncoder(enc, num_layers=cfg['n_layers'])
        self.layer_norm  = nn.LayerNorm(self.hidden_size, eps=cfg['layer_norm_eps'])
        self.dropout     = nn.Dropout(cfg['hidden_dropout_prob'])
        self.output_bias = nn.Parameter(torch.zeros(self.n_items+1))

        self._init_weights(cfg['initializer_range'])

    def _init_weights(self, std):
        for n, p in self.named_parameters():
            if 'weight' in n:
                nn.init.normal_(p, mean=0.0, std=std)
            elif 'bias' in n:
                nn.init.constant_(p, 0.0)

    def forward(self, input_ids):
        B, L = input_ids.size()
        # position embedding
        pos = torch.arange(L, device=input_ids.device).unsqueeze(0).expand(B, L)
        x = self.item_emb(input_ids) + self.pos_emb(pos)
        x = self.layer_norm(x)
        x = self.dropout(x)

        # causal mask (upper triangular)
        mask = torch.triu(torch.ones((L, L), device=x.device), diagonal=1).bool()
        h = self.encoder(x, mask=mask)

        # 최종 logits: (B, L, V)
        logits = torch.matmul(h, self.item_emb.weight.t()) + self.output_bias
        return logits

# ==================== 4. Config & W&B 초기화 ====================
config = {
    'n_layers':4, 'n_heads':4, 'hidden_size':64,
    'inner_size':256, 'hidden_dropout_prob':0.5,
    'hidden_act':'gelu', 'layer_norm_eps':1e-12,
    'initializer_range':0.02,
    'max_seq_length':20,
    'n_items': len(token2id)
}

# ==================== 5. Leave-two-out 분할 및 DataLoaders 설정 ====================
# 5.1) Leave-two-out 방식으로 train_seqs, val_seqs, test_seqs 생성
train_seqs = []
val_seqs   = []
test_seqs  = []

for seq in valid_seqs:
    # 길이가 3 이상이므로, 마지막 두 개를 val/test으로 할당
    # train: seq[:-2], val: seq[:-1], test: seq (원본 전체)
    train_seqs.append(seq[:-2])  # 마지막 두 개 제외
    val_seqs.append(seq[:-1])    # 마지막 하나 제외 (validation 에서는 penultimate 예측)
    test_seqs.append(seq)        # 원본 전체 (test 에서는 마지막 아이템 예측)

# 5.2) Dataset & DataLoader 생성
BATCH_SIZE = 128

# ─ Train DataLoader ─
#   train_seqs마다 “마지막 아이템”을 target으로 삼아 학습
train_rec_ds     = SASRecDataset(train_seqs, token2id, max_len=config['max_seq_length'])
train_rec_loader = DataLoader(train_rec_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)

# ─ Validation DataLoader ─
#   val_seqs마다 SASRecDataset이 마지막 non-pad 위치(=원본 penultimate)만 마스킹 → 그 위치의 target 예측 평가
val_rec_ds     = SASRecDataset(val_seqs,   token2id, max_len=config['max_seq_length'])
val_rec_loader = DataLoader(val_rec_ds,     batch_size=BATCH_SIZE, shuffle=False)

# ─ Test DataLoader ─
#   test_seqs마다 SASRecDataset이 마지막 non-pad 위치(=원본 마지막)만 마스킹 → 그 위치의 target 예측 평가
test_rec_ds     = SASRecDataset(test_seqs,  token2id, max_len=config['max_seq_length'])
test_rec_loader = DataLoader(test_rec_ds,    batch_size=BATCH_SIZE, shuffle=False)

# ==================== 6. Loss, Model, Optimizer ====================
device       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_sas    = SASRec(config).to(device)
criterion_ce = nn.CrossEntropyLoss(ignore_index=0)
optimizer    = torch.optim.Adam(model_sas.parameters(), lr=1e-4)

# ==================== 7. 평가 지표 헬퍼 (evaluate_ranking) ====================
def evaluate_ranking(all_scores, all_labels, ks=[1,5,10]):
    N, V = all_scores.shape
    rank = np.argsort(-all_scores, axis=1)
    metrics = {}
    for k in ks:
        topk_indices = rank[:, :k]
        hits = np.array([1 if all_labels[i] in topk_indices[i] else 0 for i in range(N)])
        prec = hits.mean() / k
        rec  = hits.mean()
        hr   = rec
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

        dcg_list = []
        for i in range(N):
            topk_full = rank[i][:k]
            if all_labels[i] in topk_full:
                r = int(np.where(topk_full == all_labels[i])[0][0])
                dcg_i = 1.0 / np.log2(r + 2)
            else:
                dcg_i = 0.0
            dcg_list.append(dcg_i)
        ndcg = float(np.mean(dcg_list))

        metrics.update({
            f"P@{k}":   prec,
            f"R@{k}":   rec,
            f"HR@{k}":  hr,
            f"F1@{k}":  f1,
            f"nDCG@{k}": ndcg,
        })
    return metrics

# ==================== 8. Train + Validation (Last‐item loss & Rec Metrics) ====================
best_val_loss = float('inf')
best_epoch    = -1

EPOCHS = 100

for epoch in tqdm(range(1, 1 + EPOCHS), desc='Training'):
    # ---------- (1) Train (penultimate이 아니라 “train_seqs의 마지막” 예측) ----------
    model_sas.train()
    train_loss = 0.0
    for inp, slens, tgt in train_rec_loader:
        # inp: (B, L), slens: (B,), tgt: (B,)
        inp, slens, tgt = inp.to(device), slens.to(device), tgt.to(device)

        logits_full = model_sas(inp)  # (B, L, V)

        # ── train_seqs에서는 “train_seqs[i]의 마지막 아이템”을 예측해야 하므로,
        #     slens[b] - 1 위치의 logit만 사용
        B, L, V = logits_full.size()
        logits_last = torch.zeros((B, V), device=inp.device)
        for b in range(B):
            last_idx = slens[b].item() - 1
            logits_last[b] = logits_full[b, last_idx]
        # ─────────────────────────────────────────────────────────────────────────

        loss = criterion_ce(logits_last, tgt)  # (B, V) vs (B,)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_rec_loader)

    # ---------- (2) Validation Loss (penultimate 예측) ----------
    model_sas.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inp, slens, tgt in val_rec_loader:
            inp, slens, tgt = inp.to(device), slens.to(device), tgt.to(device)
            logits_full = model_sas(inp)  # (B, L, V)

            B, L, V = logits_full.size()
            logits_last = torch.zeros((B, V), device=inp.device)
            for b in range(B):
                last_idx = slens[b].item() - 1
                logits_last[b] = logits_full[b, last_idx]

            val_loss += criterion_ce(logits_last, tgt).item()
    val_loss /= len(val_rec_loader)

    # ---------- (3) Validation Rec Metrics (penultimate 예측, 순위 평가) ----------
    all_scores, all_labels = [], []
    with torch.no_grad():
        for inp, slens, tgt in val_rec_loader:
            inp, slens, tgt = inp.to(device), slens.to(device), tgt.to(device)
            logits_full = model_sas(inp)  # (B, L, V)
            B, L, V     = logits_full.shape

            for b in range(B):
                last_idx = slens[b].item() - 1
                scores = logits_full[b, last_idx].clone()  # (V,)

                # 이미 본 아이템들(= prefix)에 대해 score를 -inf 처리
                # prefix 길이는 slens[b], prefix 아이템 ID는 inp[b, -slens[b]:]
                seen_ids = set(inp[b, -slens[b]:].tolist())
                seen_ids.discard(tgt[b].item())
                for i in seen_ids:
                    if i != 0:
                        scores[i] = float('-inf')

                all_scores.append(scores.detach().cpu().numpy())
                all_labels.append(tgt[b].item())

    val_metrics = evaluate_ranking(np.stack(all_scores), np.array(all_labels))

    # ---------- (4) 최적 모델 저장 ----------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch    = epoch
        torch.save(model_sas.state_dict(), "best_sasrec_leave2out.pt")

    # ── 로그 출력 ───────────────────────────────────────────────────────────────────
    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

print(f"▶ Best Epoch: {best_epoch} | Val Loss: {best_val_loss:.4f}")

# ==================== 9. Test 평가: 저장된 최적 모델 로드 후 “마지막 아이템” 예측 ====================
# (1) 최적화된 모델 로드
model_sas.load_state_dict(torch.load("best_sasrec_leave2out.pt"))
model_sas.eval()

# (2) Test Loss (마지막 아이템 예측)
test_loss = 0.0
with torch.no_grad():
    for inp, slens, tgt in tqdm(test_rec_loader, desc='Test Loss (Last‐item)'):
        inp, slens, tgt = inp.to(device), slens.to(device), tgt.to(device)
        logits_full = model_sas(inp)  # (B, L, V)

        B, L, V = logits_full.size()
        logits_last = torch.zeros((B, V), device=inp.device)
        for b in range(B):
            last_idx = slens[b].item() - 1
            logits_last[b] = logits_full[b, last_idx]

        test_loss += criterion_ce(logits_last, tgt).item()
test_loss /= len(test_rec_loader)

# (3) Test Rec Metrics (hold-out last item, ranking 평가)
all_test_scores, all_test_labels = [], []
with torch.no_grad():
    for inp, slens, tgt in test_rec_loader:
        inp, slens, tgt = inp.to(device), slens.to(device), tgt.to(device)
        logits_full = model_sas(inp)
        B, L, V     = logits_full.shape

        for b in range(B):
            last_idx = slens[b].item() - 1
            scores = logits_full[b, last_idx].clone()

            # 이미 본 아이템(= penultimate 포함)들에 대해 score -inf 처리
            seen_ids = set(inp[b, -slens[b]:].tolist())
            seen_ids.discard(tgt[b].item())
            for i in seen_ids:
                if i != 0:
                    scores[i] = float('-inf')

            all_test_scores.append(scores.detach().cpu().numpy())
            all_test_labels.append(tgt[b].item())

test_metrics = evaluate_ranking(np.stack(all_test_scores), np.array(all_test_labels))

# ── 결과 출력 ───────────────────────────────────────────────────────────────────
print("\n===== Test 결과 (Last‐item 기준) =====")
print(f"Test Loss: {test_loss:.4f}")
print(f"HR@1   : {test_metrics['HR@1']:.4f}")
print(f"HR@5   : {test_metrics['HR@5']:.4f}")
print(f"HR@10  : {test_metrics['HR@10']:.4f}")
print(f"NDCG@5 : {test_metrics['nDCG@5']:.4f}")
print(f"NDCG@10: {test_metrics['nDCG@10']:.4f}")
print(f"MRR    : {test_metrics['P@1']:.4f}")


Training:   1%|          | 1/100 [00:04<08:06,  4.91s/it]

Epoch 01 | Train Loss: 5.2184 | Val Loss: 5.2117


Training:   2%|▏         | 2/100 [00:09<07:57,  4.87s/it]

Epoch 02 | Train Loss: 5.1965 | Val Loss: 5.1890


Training:   3%|▎         | 3/100 [00:14<07:50,  4.85s/it]

...

Epoch 99 | Train Loss: 3.5742 | Val Loss: 4.1184


Training: 100%|██████████| 100/100 [08:00<00:00,  4.81s/it]


Epoch 100 | Train Loss: 3.5757 | Val Loss: 4.1343
▶ Best Epoch: 88 | Val Loss: 4.1051


Test Loss (Last‐item): 100%|██████████| 47/47 [00:00<00:00, 108.93it/s]



===== Test 결과 (Last‐item 기준) =====
Test Loss: 4.3962
HR@1   : 0.0547
HR@5   : 0.2630
HR@10  : 0.4028
NDCG@5 : 0.1601
NDCG@10: 0.2045
MRR    : 0.0547
